# 6. Hafta | Besin Takviyesi Kullanım Verilerinin Analizi


## Çalışmanın amacı
Bu çalışmada NHANES 2021–2023 araştırmasının açık verileri kullanılarak katılımcıların son 30 gündeki besin takviyesi kullanım durumu ve takviye sayısı betimsel olarak incelenmektedir. Yaş grubu ve cinsiyete göre kullanım örüntüleri karşılaştırılacaktır.

**Önemli:** Bu çalışma tanımlayıcı bir veri analizi uygulamasıdır. Takviye kullanmanın sağlık açısından gerekli olup olmadığını değerlendirmez ve nedensellik göstermez. Kodlar gerçek CDC verisini Google Colab üzerinden indirir; veri veya analiz sonucu uydurulmamıştır.


## A. Veri tanıtımı

- **Veri seti:** National Health and Nutrition Examination Survey (NHANES), August 2021–August 2023.
- **Ana veri dosyası:** Dietary Supplement Use 30-Day – Total Dietary Supplements (`DSQTOT_L.xpt`).
- **Demografik veri dosyası:** Demographics (`DEMO_L.xpt`).
- **Kaynak:** U.S. Centers for Disease Control and Prevention (CDC), National Center for Health Statistics (NCHS).
- **Erişim:** 8 Kasım 2026.
- **Biçim:** SAS Transport (`.xpt`); Python/pandas ile okunacaktır.
- **Gözlem birimi:** Her satır bir NHANES katılımcısını temsil eder. Dosyalar `SEQN` katılımcı numarasıyla birleştirilir.
- **Temel değişkenler:** `DSD010` (son 30 günde takviye kullanımı), `DSDCOUNT` (bildirilen takviye sayısı), `RIDAGEYR` (yaş), `RIAGENDR` (cinsiyet).
- **Ölçü birimleri:** Yaş yıl; takviye sayısı adet; kullanım durumu kategoriktir.
- **Veri niteliği:** Gerçek, kamuya açık araştırma verisi. CDC dokümantasyonu ve veri kullanım koşulları geçerlidir.

**Veri kaynağı bağlantıları**
- [NHANES 2021–2023 Dietary Supplement Use – Total (DSQTOT_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DSQTOT_L.htm)
- [NHANES 2021–2023 Demographics (DEMO_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm)
- [NHANES 2021–2023 veri sayfası](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Cycle=2021-2023)

**Lisans/kullanım:** CDC/NCHS kamuya açık NHANES veri ve dokümantasyonu. Veri kullanılırken CDC'nin analitik yönergeleri ve atıf bilgileri dikkate alınmalıdır.


## B. Başlangıç: kütüphaneler ve verilerin indirilmesi

NHANES dosyaları `.xpt` biçimindedir. Aşağıdaki hücreler dosyaları doğrudan CDC'nin resmi sunucusundan indirir. Google Colab'da internet bağlantısı açık olmalıdır.


In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 30)

DATA_URLS = {
    "supplement": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DSQTOT_L.xpt",
    "demographics": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt",
}

def read_nhanes_xpt(url):
    """CDC'den XPT dosyasını indirir ve pandas DataFrame olarak okur."""
    with urlopen(url) as response:
        content = response.read()
    return pd.read_sas(BytesIO(content), format="xport")

supplement = read_nhanes_xpt(DATA_URLS["supplement"])
demographics = read_nhanes_xpt(DATA_URLS["demographics"])

print("Takviye veri dosyası boyutu:", supplement.shape)
print("Demografik veri dosyası boyutu:", demographics.shape)
print("Veriler başarıyla indirildi ve okundu.")


## C. Veri inceleme ve birleştirme

Önce sütunlar, veri türleri ve eksik değerler kontrol edilir. Daha sonra takviye ve demografik dosyalar `SEQN` üzerinden birleştirilir.


In [ ]:
print("Takviye dosyasındaki sütunlar:")
print(supplement.columns.tolist())

print("\nTakviye dosyasının ilk 5 satırı:")
display(supplement.head())

print("\nTakviye dosyasındaki eksik değerler:")
display(supplement.isna().sum().sort_values(ascending=False).head(10))

# Analiz için gerekli sütunları seç
supp_cols = ["SEQN", "DSD010", "DSDCOUNT"]
demo_cols = ["SEQN", "RIDAGEYR", "RIAGENDR"]

data = supplement[supp_cols].merge(
    demographics[demo_cols],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("\nBirleştirilmiş veri boyutu:", data.shape)
display(data.head())


### Değişkenlerin düzenlenmesi

CDC kodlamasına göre `DSD010=1` son 30 günde en az bir besin takviyesi kullanıldığını, `DSD010=2` ise kullanılmadığını gösterir. 7 ve 9 yanıtları (bilmiyor/yanıtlamadı) eksik kabul edilir. Cinsiyet değişkeni CDC kodlarına göre etiketlenir. Yaş grupları yalnızca betimsel karşılaştırma amacıyla oluşturulur.


In [ ]:
# Geçerli kullanım yanıtlarını seç
data["DSD010"] = pd.to_numeric(data["DSD010"], errors="coerce")
data["DSDCOUNT"] = pd.to_numeric(data["DSDCOUNT"], errors="coerce")
data["RIDAGEYR"] = pd.to_numeric(data["RIDAGEYR"], errors="coerce")

data["kullanim_durumu"] = data["DSD010"].map({
    1: "Kullanıyor",
    2: "Kullanmıyor"
})

data["cinsiyet"] = data["RIAGENDR"].map({
    1: "Erkek",
    2: "Kadın"
})

data["yas_grubu"] = pd.cut(
    data["RIDAGEYR"],
    bins=[0, 17, 29, 44, 59, 120],
    labels=["0–17", "18–29", "30–44", "45–59", "60+"],
    include_lowest=True
)

print("Birleştirilmiş veri boyutu:", data.shape)
print("\nKullanım durumu dağılımı (ham kod):")
display(data["DSD010"].value_counts(dropna=False).sort_index())

print("\nAnalizde kullanılacak değişkenlerdeki eksik değerler:")
display(data[["kullanim_durumu", "DSDCOUNT", "RIDAGEYR", "cinsiyet", "yas_grubu"]].isna().sum())


## D. Tanımlayıcı istatistikler ve hesaplamalar

Önce geçerli kullanım yanıtlarının sayısı ve yüzdesi hesaplanır. Takviye sayısının ortalaması yalnızca takviye kullandığını bildiren katılımcılar için hesaplanır. Kullanıcı olmayanların takviye sayısını sıfır kabul ederek ortalamaya katmak, kullanıcı başına ortalama ile toplam katılımcı ortalamasını karıştıracağı için tercih edilmemiştir.


In [ ]:
valid = data[data["kullanim_durumu"].notna()].copy()
use_counts = valid["kullanim_durumu"].value_counts()
use_percent = valid["kullanim_durumu"].value_counts(normalize=True).mul(100)

usage_summary = pd.DataFrame({
    "Katılımcı sayısı (kişi)": use_counts,
    "Yüzde (%)": use_percent.round(2)
})

print("Son 30 günlük takviye kullanım durumu:")
display(usage_summary)

users = valid[valid["kullanim_durumu"] == "Kullanıyor"].copy()
supp_count_valid = users["DSDCOUNT"].dropna()

print("Geçerli kullanım yanıtı olan katılımcı sayısı:", len(valid))
print("Takviye kullandığını bildiren katılımcı sayısı:", len(users))
print("Kullanıcılar arasında takviye sayısı toplamı (adet):", int(supp_count_valid.sum()))
print("Kullanıcı başına ortalama takviye sayısı (adet):", round(supp_count_valid.mean(), 2))
print("Kullanıcı başına medyan takviye sayısı (adet):", round(supp_count_valid.median(), 2))
print("Takviye sayısı verisi eksik kullanıcı sayısı:", int(users["DSDCOUNT"].isna().sum()))


### Yaş grubu ve cinsiyete göre kullanım

Aşağıdaki tablolar gruplar içindeki kullanım yüzdesini gösterir. Bunlar ağırlıklandırılmamış örneklem yüzdeleridir; NHANES'in ulusal nüfusu temsil eden resmi tahminleri olarak yorumlanmamalıdır.


In [ ]:
age_gender = valid.dropna(subset=["yas_grubu", "cinsiyet"]).copy()

age_table = pd.crosstab(
    age_gender["yas_grubu"],
    age_gender["kullanim_durumu"],
    normalize="index"
).mul(100).round(1)

gender_table = pd.crosstab(
    age_gender["cinsiyet"],
    age_gender["kullanim_durumu"],
    normalize="index"
).mul(100).round(1)

print("Yaş gruplarına göre kullanım yüzdesi (%):")
display(age_table)

print("Cinsiyete göre kullanım yüzdesi (%):")
display(gender_table)


## E. Grafikler ve sonuç dosyaları

En az bir açıklamalı grafik istenmektedir. Bu çalışmada iki grafik oluşturulur: kullanım durumunun genel dağılımı ve yaş gruplarına göre kullanım yüzdesi. Grafikler betimleyicidir.


In [ ]:
output_dir = Path("sonuclar")
output_dir.mkdir(exist_ok=True)

# Grafik 1: kullanım durumu
fig, ax = plt.subplots(figsize=(7, 4))
usage_plot = use_counts.reindex(["Kullanıyor", "Kullanmıyor"])
ax.bar(usage_plot.index, usage_plot.values)
ax.set_title("NHANES 2021–2023: Son 30 günde besin takviyesi kullanımı")
ax.set_xlabel("Kullanım durumu")
ax.set_ylabel("Katılımcı sayısı (kişi)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(output_dir / "grafik_1_kullanim_durumu.png", dpi=180)
plt.show()

# Grafik 2: yaş grubuna göre kullanım yüzdesi
age_use_pct = (
    age_gender.assign(kullanim=age_gender["kullanim_durumu"].eq("Kullanıyor"))
    .groupby("yas_grubu", observed=False)["kullanim"]
    .mean()
    .mul(100)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(age_use_pct.index.astype(str), age_use_pct.values)
ax.set_title("Yaş gruplarına göre besin takviyesi kullanımı")
ax.set_xlabel("Yaş grubu (yıl)")
ax.set_ylabel("Takviye kullananların oranı (%)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(output_dir / "grafik_2_yas_grubuna_gore_kullanim.png", dpi=180)
plt.show()

print("Grafikler", output_dir, "klasörüne kaydedildi.")


## F. Akademik yorum

Aşağıdaki hücre, gerçek veri indirildikten ve hesaplamalar yapıldıktan sonra temel sonuçları otomatik olarak metne döker. Bu yorum örneklemdeki betimsel bulgularla sınırlıdır.


In [ ]:
n_valid = len(valid)
n_users = int((valid["kullanim_durumu"] == "Kullanıyor").sum())
n_nonusers = int((valid["kullanim_durumu"] == "Kullanmıyor").sum())
p_users = (n_users / n_valid * 100) if n_valid else np.nan
mean_supp = supp_count_valid.mean()

print(
    f"NHANES 2021–2023 verisinde son 30 günlük takviye kullanım sorusuna "
    f"geçerli yanıt veren {n_valid:,} katılımcı bulunmaktadır. "
    f"Bunların {n_users:,} kişisi ({p_users:.1f}%) takviye kullandığını, "
    f"{n_nonusers:,} kişisi kullanmadığını bildirmiştir. "
    f"Takviye kullanan ve takviye sayısı bilgisi bulunan katılımcılarda "
    f"ortalama takviye sayısı {mean_supp:.2f} adettir. "
    f"Yaş grupları ve cinsiyete göre oranlar örneklem içindeki dağılımları "
    f"betimlemektedir; tek başına neden-sonuç ilişkisi göstermez."
)


### Sınırlılıklar

1. Veriler öz bildirime dayalıdır; hatırlama ve bildirim yanlılığı bulunabilir.
2. Bu analizde örneklem ağırlıkları uygulanmamıştır. Bu nedenle bulunan yüzdeler ABD nüfusu için ulusal prevalans tahmini değildir.
3. NHANES 2021–2023 döneminde takviye kullanım nedeni ve ürünü ne kadar süredir kullandığına ilişkin bazı sorular kaldırılmıştır. Bu nedenle bu değişkenler incelenemez.
4. Takviye kullanımı son 30 günlük dönem için değerlendirilmiştir; uzun dönemli alışkanlıkları tek başına göstermez.
5. Gözlemsel ve kesitsel veriler neden-sonuç ilişkisi kurmaya izin vermez.
6. Takviye kullanımı, bireyin klinik gereksinimi veya kullanımın uygunluğu anlamına gelmez.

## G. Sonuçları CSV, JSON ve PNG olarak kaydetme

Analiz tablosu, özet istatistikler ve grafikler `sonuclar` klasörüne kaydedilir.


In [ ]:
# Temizlenmiş analiz verisi
data.to_csv(output_dir / "analiz_verisi.csv", index=False, encoding="utf-8-sig")

# Özet istatistikleri tabloya dönüştür
summary_rows = [
    {"Metrik": "Geçerli kullanım yanıtı", "Değer": n_valid, "Birim": "kişi"},
    {"Metrik": "Takviye kullandığını bildiren", "Değer": n_users, "Birim": "kişi"},
    {"Metrik": "Takviye kullanmadığını bildiren", "Değer": n_nonusers, "Birim": "kişi"},
    {"Metrik": "Takviye kullananların oranı", "Değer": round(p_users, 2), "Birim": "%"},
    {"Metrik": "Kullanıcı başına ortalama takviye sayısı", "Değer": round(mean_supp, 2), "Birim": "adet"},
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(output_dir / "ozet.csv", index=False, encoding="utf-8-sig")
summary_df.to_json(output_dir / "ozet.json", orient="records", force_ascii=False, indent=2)

age_table.to_csv(output_dir / "yas_grubuna_gore_kullanim.csv", encoding="utf-8-sig")
gender_table.to_csv(output_dir / "cinsiyete_gore_kullanim.csv", encoding="utf-8-sig")

print("Kaydedilen dosyalar:")
for file in sorted(output_dir.iterdir()):
    print("-", file.name)


## H. Sonuç ve değerlendirme

Bu uygulamada gerçek ve kamuya açık NHANES verileri kullanılarak son 30 gündeki besin takviyesi kullanımı incelenmiştir. Veri dosyaları katılımcı numarası üzerinden birleştirilmiş; eksik değerler kontrol edilmiş; kullanım durumu, takviye sayısı, yaş grubu ve cinsiyete göre betimsel özetler oluşturulmuştur. Sonuçlar örneklem içindeki kullanım örüntülerini göstermektedir. Örneklem ağırlıkları kullanılmadığından yüzdeler ulusal düzeyde genellenmemelidir. Veri setinde kullanım nedeni gibi bazı değişkenler bulunmadığı için güvenli kullanım bilgisi veya takviye kullanma motivasyonu bu analizle ölçülememektedir.


